## Setup Connection

In [1]:
import requests
import time
import dotenv
import os
import base64
import webbrowser

In [2]:
# Startup

dotenv.load_dotenv("./../../secrets.env")

APP_KEY = os.getenv("SAXO_TESTING_APP_KEY")
REDIRECT_URL = os.getenv("REDIRECTION_URI")
APP_SECRET = os.getenv("SAXO_TESTING_APP_SECRET1")
TOKEN_URL = "https://sim.logonvalidation.net/token"
API_BASE_URL = "https://gateway.saxobank.com/sim/openapi"
AUTH_BASE_URL = "https://sim.logonvalidation.net/authorize"

setup_url = f"https://sim.logonvalidation.net/authorize?response_type=code&client_id={APP_KEY}&redirect_uri=http://localhost:2000&state=12345"

# response = requests.get(setup_url)

In [3]:
setup_url

'https://sim.logonvalidation.net/authorize?response_type=code&client_id=5a0d8e4d26e94bdf813938dc21978d27&redirect_uri=http://localhost:2000&state=12345'

In [20]:
params = {
    "response_type": "code",
    "client_id": APP_KEY,
    "state": "123",
    "redirect_uri": REDIRECT_URL
}

p = requests.Request('GET', setup_url).prepare()
print(f"Please log in here:\n{p.url}\n")
webbrowser.open(p.url)

Please log in here:
https://sim.logonvalidation.net/authorize?response_type=code&client_id=5a0d8e4d26e94bdf813938dc21978d27&redirect_uri=http://localhost:2000&state=12345



True

In [21]:
redirected_url = input("Paste the full URL you were redirected to: ")
no_beginning = redirected_url[redirected_url.index("code=")+5:]
auth_code = no_beginning[:no_beginning.index("&")]

In [22]:
auth_str = f"{APP_KEY}:{APP_SECRET}"
base64_auth = base64.b64encode(auth_str.encode()).decode()

headers = {
    "Authorization": f"Basic {base64_auth}",
    "Content-Type": "application/x-www-form-urlencoded"
}

payload = {
    "grant_type": "authorization_code",
    "code": auth_code,
    "redirect_uri": REDIRECT_URL
}

In [23]:
print("Exchanging code for tokens...")
response = requests.post(TOKEN_URL, headers=headers, data=payload)

Exchanging code for tokens...


In [27]:
token_data = response.json()

In [28]:
url = f"{API_BASE_URL}/port/v1/users/me"
access_token = token_data.get("access_token")
headers = {
    "Authorization": f"Bearer {access_token}"
}

response = requests.get(url, headers=headers)

In [33]:
response.json()

{'Active': True,
 'ClientKey': '4|10kdB0NSWjosZ5I2-I2Q==',
 'Culture': 'en-GB',
 'Language': 'en',
 'LastLoginStatus': 'Successful',
 'LastLoginTime': '2026-04-26T15:16:32.720000Z',
 'LegalAssetTypes': ['FxSpot',
  'FxForwards',
  'ContractFutures',
  'Stock',
  'StockOption',
  'Bond',
  'FuturesOption',
  'StockIndexOption',
  'Cash',
  'CfdOnStock',
  'CfdOnIndex',
  'StockIndex',
  'CfdOnEtf',
  'CfdOnEtc',
  'CfdOnEtn',
  'CfdOnFund',
  'CfdOnRights',
  'CfdOnCompanyWarrant',
  'Etf',
  'Etc',
  'Etn',
  'Fund',
  'FxSwap',
  'Rights',
  'IpoOnStock',
  'CompanyWarrant'],
 'MarketDataViaOpenApiTermsAccepted': True,
 'Name': 'Robert Fischer',
 'TimeZoneId': 26,
 'UserId': '22169168',
 'UserKey': '4|10kdB0NSWjosZ5I2-I2Q=='}

In [ ]:
def renew_token_data(token_data):
    headers = {
        "Authorization": f"Basic {base64.b64encode((f"{APP_KEY}:{APP_SECRET}").encode()).decode()}",
        "Content-Type": "application/x-www-form-urlencoded"
    }
    token_data.get("")

In [34]:
def make_api_request_selector(request_obj, session, token_data):
    if(request_obj.get("Type") == "GET"):
        return requests.get(request_obj.get("URL"), headers=request_obj.get("headers"))
    elif(request_obj.get("Type") == "POST"):
        return requests.get(request_obj.get("URL"), headers=request_obj.get("headers"))

    

In [ ]:
def make_api_request(request_obj, session, token_data):

    response = make_api_request_selector(request_obj=request_obj, session=session, token_data=token_data)

    if(response.ok()):
        return response
    else:
        token_data = renew_token_data(token_data=token_data)
        

In [14]:
response.status_code

401